# JP-1 分析：日本語A だけ（Kaggle・CPU）

本番は日本語A で終わりにしました（2026-10-17 00:30）。このノートブックは、凍結した計画の**日本語の A1〜A3** と、言語ごとの副次（S1・S2・S4〜S8・S12）を、日本語A の20問で計算します。英語は回していないので、言語の比べ（A4・主B）は出しません。GPU も Colab の CU も使いません。

- 監査：事後再生監査（行 2・10・11、1問でもずれたら止める線。日本語A のノートブックですでに計算済み）と、語の種類の参照の計算し直し（行 8）。
- 結果は Kaggle の `/kaggle/working/JP1_JA_ONLY_RESULT.json` と、Dropbox の `/stash/jp1/analysis/` に置きます。画面に出すのはラベルと数だけです（問題の中身・答え・トークンは出しません）。

**使い方**：
1. Kaggle でこのノートブックを取り込みます（File → Import Notebook → ファイルを選ぶ）。
2. 右のパネルで **Internet を On** にします（電話番号の確認が済んでいれば選べます）。アクセラレーターは **None（CPU）** のままで大丈夫です。
3. Add-ons → Secrets で、`GITHUB_TOKEN`・`DROPBOX_APP_KEY`・`DROPBOX_REFRESH_TOKEN` をこのノートブックで有効にします（`DROPBOX_REFRESH_TOKEN` が無いときは、セル2でリンクとコードの手順が出るので、エディタの中で実行してね）。
4. **Run All**（エディタの中で）。10〜20 分くらいです。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったとき（エラーの文面つき）に届きます。

In [ ]:
# Cell 1 — notifications, Kaggle secrets, the exact head of the Pork repository.
import os, re, shutil, stat, subprocess, sys, json, time, hashlib, tarfile, traceback, urllib.request
from pathlib import Path
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 分析 日本語A（Kaggle・CPU・日本語だけ・A1〜A3）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 ja analysis (Kaggle)', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg[:80])
    except Exception as e:
        print('通知は送れませんでした（分析には影響しません）：', type(e).__name__)
_SECRET_RE = re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@|sl\.[A-Za-z0-9_\-]{20,}")
def _on_error(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    if e is None:
        return
    tb = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    text = _SECRET_RE.sub('[伏せ字]', 'セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
    b = text.encode('utf-8')
    if len(b) > 3300:
        text = '…（前のほうは省略）\n' + b[-3300:].decode('utf-8', 'ignore')
    notify('⚠ 止まりました：エラーが出ています。\n\n' + text, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from kaggle_secrets import UserSecretsClient
_S = UserSecretsClient()
def secret(name):
    try:
        v = _S.get_secret(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None
EXPECTED_HEAD = "42268044796bb9bf461602fc7221eb29044d8381"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/kaggle/working/Pork")
token = secret("GITHUB_TOKEN")
assert token, "Kaggle の Secrets に GITHUB_TOKEN を入れて、このノートブックで有効にしてください（Add-ons → Secrets）"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir("/kaggle/working")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
STARTED = time.time()
print("head:", head)

In [ ]:
# Cell 2 — pinned packages; ja-A from Dropbox (hash checked); identity, items hash, units validated.
from pork import jp1 as P
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
VERSIONS = {k: md.version(k) for k in P.PACKAGES}
assert VERSIONS == P.PACKAGES, ("版が違います — Claude に見せてね", VERSIONS)
from notebook_tools import dbx
DBX_KEY, DBX_RT = secret("DROPBOX_APP_KEY"), secret("DROPBOX_REFRESH_TOKEN")
assert DBX_KEY, "Kaggle の Secrets に DROPBOX_APP_KEY を入れて、このノートブックで有効にしてください"
if DBX_RT:
    try:
        dbx.Dropbox(DBX_KEY, DBX_RT).token()
    except dbx.DropboxError as e:
        print("★ DROPBOX_REFRESH_TOKEN が使えませんでした（", type(e).__name__, "）。ここで作り直します ★"); DBX_RT = None
if not DBX_RT:
    _verifier, _url = dbx.pkce_start(DBX_KEY)
    print("① このリンクを開いて、Dropbox で「許可」を押してください："); print(_url)
    DBX_RT = dbx.pkce_finish(DBX_KEY, _verifier, input("② 表示されたコードをここに貼って Enter：").strip())
DBX = dbx.Dropbox(DBX_KEY, DBX_RT)
WORK = Path("/kaggle/working/jp1_ja_A")
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)
TAR = WORK.parent / "jp1_ja_A.tar.gz"
DBX.download("/stash/jp1/ja_A/jp1_ja_A.tar.gz", TAR)                 # raises if Dropbox's content hash differs
TAR_SHA = hashlib.sha256(TAR.read_bytes()).hexdigest()
assert TAR_SHA == "e0df2009767214037f537d7fb6c3299b6596a150318ad40076fa19d786e21146", ("日本語A の預けたファイルのハッシュが記録と違います — Claude に見せてね", TAR_SHA)
with tarfile.open(TAR) as tf:
    tf.extractall(WORK, filter="data")
ID = json.loads((WORK / "RUN_IDENTITY.json").read_text(encoding="utf-8"))
assert ID["lang"] == "ja" and ID["half"] == "A" and "resample_step" not in ID, ID.get("lang")
from pork import jp1_runner as rr
ITEMS = P.items(P.load_rows("ja"), "main")                          # MGSM: the item index only, never printed
assert P.items_sha256(ITEMS) == ID["items_sha256"], "問題のハッシュが日本語A と違います — Claude に見せてね"
CASES = rr.load_cases("ja", ID["model"], P.HALVES["A"], ITEMS, units_dir=WORK / "units", code_commit=ID["code_commit"],
                      items_hash=ID["items_sha256"])
AUD = json.loads((WORK / "AUDIT_ja_A.json").read_text(encoding="utf-8"))
print(json.dumps({"versions_ok": True, "tar_sha256_ok": True, "rows": len(CASES), "points": sum(len(c["points"]) for c in CASES),
                  "run_code_commit": ID["code_commit"], "audit_rows": AUD["rows"], "audit_diverged_rows": AUD["diverged_rows"]}))

In [ ]:
# Cell 3 — category audit (row 8), the analysis (pork.jp1_ja_only), result to /kaggle/working and Dropbox.
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
import fugashi
from pork import jp1_reference as R
from pork import jp1_ja_only as JO
from pork import jp1_amendment2 as A2
SPEC = P.MODELS[ID["model"]]
TOKDIR = Path("/kaggle/working/sarashina_tokenizer")
snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(TOKDIR), token=False,
                  allow_patterns=["tokenizer*", "*.model", "special_tokens_map.json", "config.json"])
TOK = AutoTokenizer.from_pretrained(str(TOKDIR))
_vocab = sorted(TOK.get_vocab().items())
assert hashlib.sha256(json.dumps(_vocab, ensure_ascii=False).encode("utf-8")).hexdigest() == P.TOKENIZER_VOCAB_SHA256
del _vocab
decode = lambda ids: TOK.decode(list(ids), skip_special_tokens=True)
TAGGER = fugashi.Tagger()
BY_ROW = {c["row"]: c for c in CASES}
COMPARES = {r: R.compare_base("ja", BY_ROW[r]["base"], decode, tagger=TAGGER) for r in A2.CATEGORY_AUDIT_ROWS_A2}
RES = JO.run(CASES, AUD, COMPARES, head=EXPECTED_HEAD,
             sources={"tar": "/stash/jp1/ja_A/jp1_ja_A.tar.gz", "tar_sha256": TAR_SHA, "run_code_commit": ID["code_commit"],
                      "versions": VERSIONS})
RES["elapsed_minutes"] = (time.time() - STARTED) / 60
OUT = Path("/kaggle/working/JP1_JA_ONLY_RESULT.json")
OUT.write_text(json.dumps(RES, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
DBX.upload(str(OUT), "/stash/jp1/analysis/JP1_JA_ONLY_RESULT.json")
r = RES["per_lang"]["ja"]
print(json.dumps({"labels": RES["labels"], "replay_audit_pass": RES["replay_audit"]["pass"],
                  "category_audit_pass": RES["category_audit"]["pass"], "base_accuracy": r["base_accuracy"],
                  "guard_ok": r["guard_ok"], "items_in_analysis": r["S8_lengths"]["items_in_analysis"],
                  "positions": r["S8_lengths"]["positions"], "result_sha256": RES["result_sha256"]}, indent=1, ensure_ascii=False))
notify("🎉 日本語A の分析 終わりました（結果は Dropbox の /stash/jp1/analysis/）", tags='tada')